# 🤖 Vision-Language Grounding 
## Mapping Language to Objects in a Scene


In this tutorial, you'll learn how to build a system that understands **what objects people are talking about** when they describe things in a scene.

**Real-world example:**
- You say: "Pick up the red mug"
- The robot needs to figure out: Which mug? Where is it? What does 'red' mean?

This is called **Vision-Language Grounding** - connecting language (words) to vision (what we see).

---

### What You'll Build:
A system that:
1. 📸 Takes an image with multiple objects
2. 🗣️ Accepts a language description (like "the blue cup")
3. 🎯 Identifies which object you're referring to
4. 📊 Gives you confidence scores and attributes

### What You'll Learn:
- What is CLIP? (A vision-language model)
- How to compare images and text
- How to structure predictions with confidence scores
- How to handle multiple objects in a scene

**Don't worry!** Everything will be explained step-by-step. Let's get started! 🚀

---
## Part 1: Understanding the Problem

### Imagine This Scenario:

You have a table with:
- A red mug (let's call it `mug_1`)
- A blue mug (let's call it `mug_2`)
- A green plate (let's call it `plate_1`)

Someone says: **"Pick up the blue container"**

### The Challenge:
1. Which object is "blue"? → `mug_2` ✓
2. Which objects are "containers"? → `mug_1` and `mug_2` (mugs hold things)
3. Which one is BOTH blue AND a container? → `mug_2` ✓✓

### What We Need:
```
Output:
  - object: mug_2
  - confidence: 0.83  (83% sure this is correct)
  - attributes: {container, movable, blue}
```

That's what we're building! 🎯

---
## ⚠️ Part 2: CRITICAL SETUP (RUN THIS FIRST!) ⚠️


**⚡ YOU MUST RUN THE CELL BELOW EVERY TIME YOU:**
- Restart the kernel
- Restart the notebook
- Restart your instance/container

### ❓ Why This Cell?
The `torchao` package (required by unsloth_zoo) conflicts with transformers and causes this error:
```
ImportError: cannot import name 'CLIPProcessor' from 'transformers'
```

### ✅ What This Cell Does:
1. Removes torchao automatically (takes ~10-30 seconds)
2. Verifies all required packages are installed
3. Shows "✨ ENVIRONMENT READY!" when done

### 📋 Instructions:
1. ▶️ Click the code cell below
2. ⌨️ Press **Shift+Enter** (or click Run)
3. ⏳ Wait for "✨ ENVIRONMENT READY!" message
4. ✅ Continue to Part 3 (imports)

**This is your permanent fix - just run it after each restart!** 🚀

In [2]:
# ⚠️⚠️⚠️ CRITICAL: RUN THIS CELL FIRST AFTER EVERY RESTART! ⚠️⚠️⚠️
# This cell fixes the torchao conflict that blocks transformers imports

import subprocess
import sys

print("="*70)
print("🔧 FIXING ENVIRONMENT - Please wait...")
print("="*70)

# Remove torchao
print("\n[1/3] Removing incompatible torchao package...")
try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "torchao", "-y"],
        capture_output=True, text=True, check=False, timeout=60
    )
    if "Successfully uninstalled" in result.stdout:
        print("      ✅ torchao removed!")
    else:
        print("      ℹ️  torchao not found (OK)")
except Exception as e:
    print(f"      ⚠️  {e}")

# Verify environment
print("\n[2/3] Verifying Python environment...")
print(f"      Python: {sys.version.split()[0]}")

# Check packages
print("\n[3/3] Checking required packages...")
packages = [("transformers", "transformers"), ("torch", "torch"), 
            ("Pillow", "PIL"), ("numpy", "numpy"), ("pydantic", "pydantic")]

for pkg_name, import_name in packages:
    try:
        __import__(import_name)
        print(f"      ✅ {pkg_name}")
    except ImportError:
        print(f"      📦 Installing {pkg_name}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pkg_name, "-q"],
                      capture_output=True, timeout=120)

# Success!
print("\n" + "="*70)
print("✨ ENVIRONMENT READY! You can now run the next cells.")
print("="*70)
print("\n💡 Next step: Run Part 3 (imports) below")
print("="*70)

🔧 FIXING ENVIRONMENT - Please wait...

[1/3] Removing incompatible torchao package...
      ℹ️  torchao not found (OK)

[2/3] Verifying Python environment...
      Python: 3.12.12

[3/3] Checking required packages...
      ✅ transformers
      ✅ torch
      ✅ Pillow
      ✅ numpy
      ✅ pydantic

✨ ENVIRONMENT READY! You can now run the next cells.

💡 Next step: Run Part 3 (imports) below


---
## Part 3: Installing and Importing Libraries

First, we need to install some tools. Think of these as adding special abilities to Python.

### Why each library?
- **transformers**: Gives us access to CLIP (the AI model)
- **torch**: The "engine" that runs AI models
- **Pillow**: For working with images
- **numpy**: For working with numbers and arrays
- **pydantic**: For creating structured data (clean organization!)

In [3]:
# Install required libraries

!pip install transformers torch Pillow numpy pydantic -q

print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


In [4]:
# Import the libraries we'll use
# Think of 'import' like opening a toolbox and taking out specific tools

from transformers import CLIPProcessor, CLIPModel  # The AI model for vision-language
from PIL import Image, ImageDraw, ImageFont  # For working with images
import numpy as np  # For math operations on arrays
from typing import List, Dict, Optional  # For type hints (helps us write cleaner code)
from pydantic import BaseModel, Field  # For structured data
import json  # For working with JSON data


print("✅ All imports successful! Ready to code!")

✅ All imports successful! Ready to code!


---
## Part 4: What is CLIP?

### CLIP = Contrastive Language-Image Pre-training

**In Simple Terms:**
CLIP is an AI model that understands BOTH images AND text. It was trained on millions of image-text pairs from the internet.

### How CLIP Works:
1. You give it an image → It creates a "fingerprint" (embedding)
2. You give it text → It creates another "fingerprint" (embedding)
3. It compares them: Similar fingerprints = related image and text! ✨

### Visual Explanation:
```
Image: [🔵 Blue Mug Photo]  →  CLIP  →  [0.2, 0.8, 0.1, ...]  (numbers representing the image)
Text:  "blue mug"          →  CLIP  →  [0.3, 0.7, 0.2, ...]  (numbers representing the text)
                                          ↓
                                    Compare these!
                                          ↓
                                    Similarity: 0.92 (very similar!) ✓
```

### Why CLIP is Perfect for Our Task:
- It can match "red mug" (text) to actual red mug images
- It understands colors, shapes, and object types
- It's pre-trained, so we don't need to train it ourselves! 🎉

In [ ]:
# Load CLIP model
# This is like opening the AI "brain" that will help us understand images and text

print("Loading CLIP model... (this might take a minute)")

# Load the model and processor
# Model = the AI brain
# Processor = prepares data for the brain to understand
# use_safetensors=True ensures safe loading with PyTorch < 2.6
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("✅ CLIP model loaded successfully!")
print("Now we can compare images and text!")

Loading CLIP model... (this might take a minute)
✅ CLIP model loaded successfully!
Now we can compare images and text!


---
## Part 4: Creating Data Structures

### Why Do We Need Structures?

Imagine trying to organize information on loose papers vs. using labeled folders. Structures help us keep data organized!

### What We'll Create:
1. **ObjectHypothesis**: One prediction about which object is being referred to
2. **SceneObject**: Information about an object in the scene
3. **GroundingResult**: The final answer with all predictions

We use **Pydantic** because it:
- Validates our data (makes sure everything is correct)
- Provides clear structure
- Helps prevent bugs

In [ ]:
# Define our data structures using Pydantic
# Think of these as "templates" or "forms" that our data must follow

class ObjectHypothesis(BaseModel):
    """
    Represents one prediction about which object is being referred to.
    
    Example:
        "I think you mean 'mug_2' with 83% confidence"
    """
    #The three dots (...) mean: "This field is REQUIRED"
    
    object_id: str = Field(..., description="Unique identifier for the object (e.g., 'mug_2')")
    confidence: float = Field(..., ge=0.0, le=1.0, description="How confident we are (0.0 to 1.0)")
    attributes: List[str] = Field(default_factory=list, description="Properties of the object (e.g., 'blue', 'container')")
    reasoning: Optional[str] = Field(None, description="Why we think this is the correct object")
    #The inner Config class is NOT a regular nested class - it's a special configuration class that Pydantic uses to customize behavior.
    #This tells Pydantic: "Hey, when someone asks for an example of this class, show them this!"
    #It's like having a sample filled-out form so people know what to put in each field.
    class Config:
        # This allows us to create examples easily and when someone prints the object, it will show this example
        json_schema_extra = {
            "example": {
                "object_id": "mug_2",
                "confidence": 0.83,
                "attributes": ["container", "movable", "blue"],
                "reasoning": "Object matches 'blue' attribute and is a container type"
            }
        }


class SceneObject(BaseModel):
    """
    Represents one object in the scene.
    
    This is what we KNOW about an object (before language grounding).
    """
    object_id: str = Field(..., description="Unique identifier (e.g., 'mug_1', 'plate_1')")
    object_type: str = Field(..., description="Type of object (e.g., 'mug', 'plate', 'cup')")
    image_crop: Optional[str] = Field(None, description="Path to cropped image of this object")
    bounding_box: Optional[Dict[str, int]] = Field(None, description="Location in image {x, y, width, height}")
    attributes: List[str] = Field(default_factory=list, description="Known attributes (e.g., 'movable')")


class GroundingResult(BaseModel):
    """
    The final result: all our predictions ranked by confidence.
    """
    query: str = Field(..., description="The original language description")
    hypotheses: List[ObjectHypothesis] = Field(..., description="List of predictions, best first")
    top_match: Optional[str] = Field(None, description="The most likely object_id")
    
    def get_top_match(self) -> Optional[ObjectHypothesis]:
        """Get the hypothesis with highest confidence."""
        if self.hypotheses:
            return self.hypotheses[0]  # List is already sorted by confidence
        return None


print("✅ Data structures defined!")
print("Now we can organize our predictions in a clean, structured way.")

✅ Data structures defined!
Now we can organize our predictions in a clean, structured way.


/tmp/ipykernel_6830/3881537770.py:4: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class ObjectHypothesis(BaseModel):


### Let's Test Our Structures!

Before building the full system, let's make sure our data structures work correctly.

In [ ]:
# Create a test hypothesis
# This is like filling out our "form" with actual data

test_hypothesis = ObjectHypothesis(
    object_id="mug_2",
    confidence=0.83,
    attributes=["container", "movable", "blue"],
    reasoning="This mug is blue and can hold liquids"
)

print("📋 Test Hypothesis Created:")
print(json.dumps(test_hypothesis.model_dump(), indent=2))
print("\n✅ Our data structure works perfectly!")

📋 Test Hypothesis Created:
{
  "object_id": "mug_2",
  "confidence": 0.83,
  "attributes": [
    "container",
    "movable",
    "blue"
  ],
  "reasoning": "This mug is blue and can hold liquids"
}

✅ Our data structure works perfectly!


---
## Part 5: Building the Vision-Language Grounder

### What is a "Grounder"?
A grounder is a system that **grounds** (connects) language to the real world. In our case, it connects words to objects in images.

### Our Grounder Will:
1. Take a language query (e.g., "the blue mug")
2. Compare it to all objects in the scene
3. Score each object (how well does it match?)
4. Return ranked results

### The Magic Happens Here! ✨

In [ ]:
#This class has not been defined with the Pydantic structure
#The reason is that the Pydantic structure is recommended for data validation and when you have to store information
#If you need to do actions and perform operations, you can use a regular class
class VisionLanguageGrounder:
    """
    This class does the main work: matching language to objects.
    
    Think of it as a smart assistant that:
    1. Listens to what you say
    2. Looks at all the objects
    3. Finds the best match
    """
    
    def __init__(self, model, processor):
        """
        Initialize the grounder with a CLIP model.
        
        Args:
            model: The CLIP model (the AI brain)
            processor: The CLIP processor (prepares data)
        """
        self.model = model
        self.processor = processor
        
        # Pre-defined attributes we can detect
        # We can extend this list based on what CLIP understands well
        self.known_attributes = [
            "red", "blue", "green", "yellow", "black", "white",  # Colors
            "large", "small", "tall", "short",  # Sizes
            "container", "movable", "fragile", "heavy"  # Properties
        ]
    
    def compute_similarity(self, image, text: str) -> float:
        """
        Compare an image to text and return a similarity score.
        
        This is the CORE function! It uses CLIP to compare.
        
        Args:
            image: A PIL Image object
            text: A string description
        
        Returns:
            A number between 0 and 1 (higher = more similar)
        
        Example:
            score = compute_similarity(blue_mug_image, "blue mug")
            # Returns: 0.92 (very similar!)
        """
        # Prepare the image and text for CLIP
        # The processor converts them into numbers the model can understand
        inputs = self.processor(
            text=[text],  # Text must be in a list
            images=image,  # The image
            return_tensors="pt",  # Return PyTorch tensors
            padding=True  # Make sure everything is the same size
        )
        
        # Get predictions from the model
        # This is where the AI "thinks"
        outputs = self.model(**inputs)
        
        # Get the similarity score
        # logits_per_image[0][0] gives us the match score
        logits_per_image = outputs.logits_per_image
        similarity = logits_per_image.softmax(dim=1)[0][0].item()
        
        return similarity
    
    def detect_attributes(self, image, object_type: str) -> List[str]:
        """
        Detect attributes of an object from its image.
        
        This function checks which attributes (color, size, etc.) match the object.
        
        Args:
            image: A PIL Image of the object
            object_type: Type of object (e.g., "mug")
        
        Returns:
            List of detected attributes (e.g., ["blue", "container"])
        
        How it works:
            For each attribute, we ask CLIP:
            "Does this image match 'blue mug'?"
            If score > threshold, we say yes!
        """
        detected = []
        threshold = 0.22  # How confident we need to be (22%)
        
        # Check each possible attribute
        for attr in self.known_attributes:
            # Create a description with this attribute
            text_query = f"{attr} {object_type}"
            
            # Compare image to description
            score = self.compute_similarity(image, text_query)
            
            # If match is strong enough, add this attribute
            if score > threshold:
                detected.append(attr)
        
        return detected
    
    def ground_query(
        self,
        query: str,
        scene_objects: List[SceneObject],
        top_k: int = 3
    ) -> GroundingResult:
        """
        Main function: Ground a language query to objects in the scene.
        
        This is where everything comes together!
        
        Args:
            query: Language description (e.g., "the blue container")
            scene_objects: List of objects in the scene
            top_k: How many top matches to return (default: 3)
        
        Returns:
            GroundingResult with ranked hypotheses
        
        Process:
            1. For each object in scene:
                a. Load its image
                b. Compare to query
                c. Detect its attributes
            2. Rank all objects by similarity
            3. Return top matches
        """
        hypotheses = []
        
        print(f"\n🔍 Grounding query: '{query}'")
        print(f"Comparing against {len(scene_objects)} objects...\n")
        
        # Evaluate each object in the scene
        for scene_obj in scene_objects:
            # Skip if no image available
            if not scene_obj.image_crop:
                print(f"⚠️  Skipping {scene_obj.object_id}: No image available")
                continue
            
            # Load the object's image
            try:
                image = Image.open(scene_obj.image_crop).convert("RGB")
            except Exception as e:
                print(f"⚠️  Error loading {scene_obj.object_id}: {e}")
                continue
            
            # Compute similarity between query and this object
            confidence = self.compute_similarity(image, query)
            
            # Detect attributes of this object
            detected_attrs = self.detect_attributes(image, scene_obj.object_type)
            
            # Combine with any pre-known attributes
            all_attrs = list(set(detected_attrs + scene_obj.attributes))
            
            # Create reasoning explanation
            reasoning = f"Similarity score: {confidence:.3f}. Detected attributes: {', '.join(all_attrs) if all_attrs else 'none'}."
            
            # Create hypothesis for this object
            hypothesis = ObjectHypothesis(
                object_id=scene_obj.object_id,
                confidence=confidence,
                attributes=all_attrs,
                reasoning=reasoning
            )
            
            hypotheses.append(hypothesis)
            
            print(f"  {scene_obj.object_id}: confidence={confidence:.3f}, attributes={all_attrs}")
        
        # Sort hypotheses by confidence (highest first)
        hypotheses.sort(key=lambda h: h.confidence, reverse=True)
        
        # Keep only top_k matches
        top_hypotheses = hypotheses[:top_k]
        
        # Create final result
        result = GroundingResult(
            query=query,
            hypotheses=top_hypotheses,
            top_match=top_hypotheses[0].object_id if top_hypotheses else None
        )
        
        print(f"\n✅ Top match: {result.top_match} (confidence: {top_hypotheses[0].confidence:.3f})")
        
        return result


print("✅ VisionLanguageGrounder class created!")
print("This is the heart of our system!")

✅ VisionLanguageGrounder class created!
This is the heart of our system!


---
## Part 6: Creating Demo Data

Since we might not have real images available, let's create synthetic (fake) images for demonstration. This will help you understand the system even without a camera!

### What We'll Create:
- Simple colored rectangles representing objects
- A scene with 3 objects: red mug, blue mug, green plate

In a real application, you would use actual photos or video frames from a robot's camera.

In [ ]:
import os

# Create a directory for our demo images
os.makedirs("demo_objects", exist_ok=True)

def create_demo_object_image(color: str, object_type: str, filename: str):
    """
    Create a simple colored rectangle to represent an object.
    
    In real use, you'd use actual photos. But for learning, this works great!
    
    Args:
        color: Color name (e.g., "red")
        object_type: Type of object (e.g., "mug")
        filename: Where to save the image
    """
    # Color mapping (RGB values)
    color_map = {
        "red": (220, 50, 50),
        "blue": (50, 100, 220),
        "green": (50, 200, 80),
        "yellow": (240, 220, 50),
    }
    
    # Create a 200x200 pixel image
    img = Image.new('RGB', (200, 200), color=color_map.get(color, (128, 128, 128)))
    draw = ImageDraw.Draw(img)
    
    # Add text label
    text = f"{color}\n{object_type}"
    # Draw text in white
    draw.text((10, 10), text, fill=(255, 255, 255))
    
    # Save the image
    img.save(filename)
    print(f"  Created: {filename}")


# Create demo object images
print("Creating demo object images...")
create_demo_object_image("red", "mug", "demo_objects/mug_1.png")
create_demo_object_image("blue", "mug", "demo_objects/mug_2.png")
create_demo_object_image("green", "plate", "demo_objects/plate_1.png")
create_demo_object_image("yellow", "cup", "demo_objects/cup_1.png")

print("\n✅ Demo images created!")
print("In a real robot system, these would be actual photos of objects.")

Creating demo object images...
  Created: demo_objects/mug_1.png
  Created: demo_objects/mug_2.png
  Created: demo_objects/plate_1.png
  Created: demo_objects/cup_1.png

✅ Demo images created!
In a real robot system, these would be actual photos of objects.


### Create Scene Objects

Now let's define what objects exist in our scene using the `SceneObject` structure we created earlier.

In [ ]:
# Define our scene: what objects exist and where they are
scene_objects = [
    SceneObject(
        object_id="mug_1",
        object_type="mug",
        image_crop="demo_objects/mug_1.png",
        attributes=["container", "movable"]  # Mugs can hold things and be moved
    ),
    SceneObject(
        object_id="mug_2",
        object_type="mug",
        image_crop="demo_objects/mug_2.png",
        attributes=["container", "movable"]
    ),
    SceneObject(
        object_id="plate_1",
        object_type="plate",
        image_crop="demo_objects/plate_1.png",
        attributes=["movable"]  # Plates are movable but not containers
    ),
    SceneObject(
        object_id="cup_1",
        object_type="cup",
        image_crop="demo_objects/cup_1.png",
        attributes=["container", "movable", "fragile"]
    ),
]

print("✅ Scene setup complete!")
print(f"\nOur scene has {len(scene_objects)} objects:")
for obj in scene_objects:
    print(f"  - {obj.object_id}: {obj.object_type} with attributes {obj.attributes}")

✅ Scene setup complete!

Our scene has 4 objects:
  - mug_1: mug with attributes ['container', 'movable']
  - mug_2: mug with attributes ['container', 'movable']
  - plate_1: plate with attributes ['movable']
  - cup_1: cup with attributes ['container', 'movable', 'fragile']


---
## Part 7: Testing the System! 🎯

Now comes the exciting part - let's see our system in action!

We'll test different language queries and see how well it identifies objects.

In [ ]:
# Create our grounder
grounder = VisionLanguageGrounder(model, processor)

print("🚀 Vision-Language Grounder is ready!")
print("Let's test it with some queries...")

🚀 Vision-Language Grounder is ready!
Let's test it with some queries...


### Test 1: Simple Color Query

In [ ]:
# Test Query 1: "the blue mug"
query1 = "the blue mug"

print("="*60)
print(f"TEST 1: Query = '{query1}'")
print("="*60)

result1 = grounder.ground_query(query1, scene_objects)

print("\n📊 Results:")
print(json.dumps(result1.model_dump(), indent=2))

### Test 2: Type Query

In [ ]:
# Test Query 2: "a container"
query2 = "a container"

print("="*60)
print(f"TEST 2: Query = '{query2}'")
print("="*60)

result2 = grounder.ground_query(query2, scene_objects)

print("\n📊 Results:")
print(json.dumps(result2.model_dump(), indent=2))

### Test 3: Combined Attributes

In [ ]:
# Test Query 3: "the red container"
query3 = "the red container"

print("="*60)
print(f"TEST 3: Query = '{query3}'")
print("="*60)

result3 = grounder.ground_query(query3, scene_objects)

print("\n📊 Results:")
print(json.dumps(result3.model_dump(), indent=2))

### Test 4: Your Turn! ✏️

Now try your own query! Change the text below and run the cell.

In [ ]:
# YOUR CUSTOM QUERY HERE!
# Try different descriptions like:
# - "the green object"
# - "the yellow cup"
# - "a movable item"

your_query = "the green plate"  # ← Change this!

print("="*60)
print(f"YOUR TEST: Query = '{your_query}'")
print("="*60)

your_result = grounder.ground_query(your_query, scene_objects)

print("\n📊 Results:")
print(json.dumps(your_result.model_dump(), indent=2))

---
## Part 8: Understanding the Results

Let's break down what the system is telling us:

### Confidence Score:
- **0.0 - 0.3**: Low confidence (probably not the right object)
- **0.3 - 0.6**: Medium confidence (could be it)
- **0.6 - 1.0**: High confidence (very likely the right one!)

### Why Multiple Hypotheses?
Sometimes language is ambiguous! For example:
- "Pick up the container" → Could be any mug or cup
- The system shows ALL possibilities, ranked by confidence
- You can use the top match, or show options to the user

### Attributes:
The system detects:
- **Colors**: red, blue, green, etc.
- **Properties**: container, movable, fragile, etc.
- **Sizes**: large, small (if distinguishable)

This helps explain WHY an object was chosen!

---
## Part 9: Practical Helper Functions

Let's create some helper functions to make working with results easier!

In [ ]:
def print_grounding_result(result: GroundingResult):
    """
    Print the grounding result in a beautiful, readable format.
    
    This makes it easier to understand what the system found.
    """
    print("\n" + "="*60)
    print(f"🔍 QUERY: '{result.query}'")
    print("="*60)
    
    if not result.hypotheses:
        print("❌ No matches found!")
        return
    
    print(f"\n🏆 TOP MATCH: {result.top_match}\n")
    
    print("📊 All Hypotheses (ranked by confidence):\n")
    
    for i, hyp in enumerate(result.hypotheses, 1):
        # Create visual confidence bar
        bar_length = int(hyp.confidence * 20)  # Scale to 20 chars
        confidence_bar = "█" * bar_length + "░" * (20 - bar_length)
        
        print(f"  {i}. {hyp.object_id}")
        print(f"     Confidence: [{confidence_bar}] {hyp.confidence:.2%}")
        print(f"     Attributes: {', '.join(hyp.attributes) if hyp.attributes else 'none'}")
        print(f"     Reasoning: {hyp.reasoning}")
        print()


def get_best_match_with_threshold(result: GroundingResult, threshold: float = 0.3) -> Optional[str]:
    """
    Get the best match only if confidence is above a threshold.
    
    This prevents returning low-confidence matches that are probably wrong.
    
    Args:
        result: The grounding result
        threshold: Minimum confidence (default 0.3 = 30%)
    
    Returns:
        object_id if confident enough, None otherwise
    """
    top = result.get_top_match()
    if top and top.confidence >= threshold:
        return top.object_id
    return None


print("✅ Helper functions created!")
print("Now let's use them to display results nicely...")

In [ ]:
# Let's use our beautiful print function!
print_grounding_result(result1)

# Check if we're confident enough
best_match = get_best_match_with_threshold(result1, threshold=0.2)
if best_match:
    print(f"✅ System is confident: Best match is {best_match}")
else:
    print("⚠️  System is not confident enough to make a decision")

---
## Part 10: Real-World Applications 🌍

### Where Can You Use This?

1. **Robotics**
   - "Pick up the red block" → Robot identifies and grasps it
   - "Move the blue container to the shelf" → Robot understands both objects

2. **Smart Home**
   - "Turn off the lamp on the left" → System identifies which lamp
   - "Find my black phone" → Searches through camera feeds

3. **Augmented Reality**
   - Point at object: "What is this?" → System identifies and explains
   - "Highlight all the cups" → AR overlays appear on cups

4. **Inventory Management**
   - "Count all the blue items" → Automatically identifies and counts
   - "Where is the large screwdriver?" → Locates it in warehouse

5. **Accessibility**
   - Helping visually impaired: "What's in front of me?"
   - "Find the medicine bottle" → Identifies and guides user

---
## Part 11: Limitations & Improvements 🔧

### Current Limitations:

1. **Simple Demo Images**
   - We used colored rectangles
   - Real objects are more complex
   - **Solution**: Use actual photos or camera feeds

2. **Limited Attributes**
   - We only check predefined attributes
   - Can't detect "rusty", "new", "broken", etc.
   - **Solution**: Expand attribute list or use GPT-4V for open-ended descriptions

3. **No Spatial Reasoning**
   - Can't understand "the mug to the left of the plate"
   - No concept of object relationships
   - **Solution**: Add spatial reasoning module with object positions

4. **No Context Memory**
   - Can't handle: "Pick up the mug. Now move it left."
   - Doesn't remember previous references
   - **Solution**: Add dialogue state tracking

### Possible Improvements:

```python
# 1. Add spatial relationships
class SpatialRelation(BaseModel):
    relation_type: str  # "left_of", "above", "near", etc.
    reference_object: str

# 2. Add confidence thresholds per task
class ConfidenceConfig:
    grasping: float = 0.7  # Need high confidence to grasp
    pointing: float = 0.4  # Can point with lower confidence

# 3. Use GPT-4V for complex descriptions
def detect_complex_attributes(image, object_type):
    # Send image to GPT-4V with prompt
    prompt = f"Describe this {object_type} in detail."
    # Returns: "A worn red ceramic mug with a handle"
```

---
## Part 12: Exercises  ✏️

Try these challenges to deepen your understanding:

### Easy:
1. Add a new demo object (e.g., a purple bowl)
2. Test the query: "the purple object"
3. Add a new attribute to the `known_attributes` list

### Medium:
4. Modify `detect_attributes()` to have different thresholds for colors vs. properties
5. Add a function to filter hypotheses by specific attributes
6. Create a function that explains why top_match was chosen

### Hard:
7. Add support for "or" queries: "the red or blue mug"
8. Implement a confidence adjustment based on attribute matches
9. Add a visualization that shows all objects with their confidence scores

### Space for Your Experiments:

In [ ]:
# YOUR CODE HERE!
# Try the exercises above, or experiment with your own ideas



---
## Part 13: Saving Results

Let's learn how to save our grounding results to files for later use.

In [ ]:
def save_grounding_result(result: GroundingResult, filename: str):
    """
    Save a grounding result to a JSON file.
    
    This is useful for:
    - Keeping logs of robot decisions
    - Analyzing performance later
    - Debugging when things go wrong
    """
    with open(filename, 'w') as f:
        json.dump(result.model_dump(), f, indent=2)
    print(f"💾 Saved result to: {filename}")


def load_grounding_result(filename: str) -> GroundingResult:
    """
    Load a grounding result from a JSON file.
    """
    with open(filename, 'r') as f:
        data = json.load(f)
    return GroundingResult(**data)


# Save our first result
save_grounding_result(result1, "result_blue_mug.json")

# Load it back
loaded_result = load_grounding_result("result_blue_mug.json")
print(f"\n✅ Loaded result for query: '{loaded_result.query}'")
print(f"   Top match: {loaded_result.top_match}")

---
## Part 14: Summary & Key Takeaways 🎓

### What You Learned:

1. **Vision-Language Grounding**: Connecting words to objects in images

2. **CLIP Model**: An AI that understands both images and text
   - Compares image "fingerprints" to text "fingerprints"
   - Returns similarity scores (0 to 1)

3. **Structured Data**: Using Pydantic to organize results
   - `ObjectHypothesis`: One prediction
   - `SceneObject`: One object in scene
   - `GroundingResult`: Final answer with rankings

4. **Confidence Scores**: How sure the system is
   - Use thresholds to avoid bad matches
   - Return multiple options when uncertain

5. **Attribute Detection**: Finding object properties
   - Colors, sizes, types
   - Helps explain decisions

### Key Code Pattern:

```python
# 1. Define objects in scene
scene_objects = [SceneObject(...), ...]

# 2. Create grounder with CLIP
grounder = VisionLanguageGrounder(model, processor)

# 3. Ground a language query
result = grounder.ground_query("the blue mug", scene_objects)

# 4. Use the result
if result.get_top_match().confidence > 0.5:
    print(f"Found: {result.top_match}")
```

### Next Steps:

1. **Try with real images**: Use your phone camera!
2. **Extend attributes**: Add more properties to detect
3. **Add spatial reasoning**: Handle "left of", "above", etc.
4. **Integrate with a robot**: Make it move objects!
5. **Use GPT-4V**: For more complex descriptions

### Resources to Learn More:

- CLIP Paper: https://arxiv.org/abs/2103.00020
- Hugging Face Transformers: https://huggingface.co/docs/transformers
- Pydantic Documentation: https://docs.pydantic.dev
- Vision-Language Research: https://paperswithcode.com/task/visual-grounding

---
## 🎉 Congratulations!

You've built a complete vision-language grounding system from scratch!

You now understand:
- ✅ How AI connects language to vision
- ✅ How to use CLIP for image-text matching
- ✅ How to structure predictions with confidence scores
- ✅ How to handle multiple objects and attributes

This is a fundamental skill for:
- 🤖 Robotics
- 🏠 Smart home systems
- 🎮 AR/VR applications
- ♿ Accessibility tools



---
## Bonus: Complete End-to-End Example

Here's a complete example putting everything together:

In [ ]:
# COMPLETE END-TO-END EXAMPLE

def complete_vision_grounding_demo():
    """
    A complete demonstration of the entire system.
    """
    print("\n" + "="*70)
    print("🤖 COMPLETE VISION-LANGUAGE GROUNDING DEMO")
    print("="*70 + "\n")
    
    # Test queries
    test_queries = [
        "the blue mug",
        "the red container",
        "a green object",
        "the yellow cup",
        "a movable container",
    ]
    
    # Run all tests
    results = []
    for query in test_queries:
        result = grounder.ground_query(query, scene_objects, top_k=2)
        results.append(result)
        print_grounding_result(result)
        print("\n" + "-"*70 + "\n")
    
    # Summary
    print("\n" + "="*70)
    print("📊 SUMMARY")
    print("="*70 + "\n")
    
    for i, (query, result) in enumerate(zip(test_queries, results), 1):
        top = result.get_top_match()
        if top:
            print(f"{i}. '{query}' → {top.object_id} (confidence: {top.confidence:.2%})")
        else:
            print(f"{i}. '{query}' → No confident match")
    
    print("\n✅ Demo complete!")


# Run the complete demo
complete_vision_grounding_demo()